# Netflix Data Analysis — Exploratory Data Analysis
**Dataset:** Netflix Movies and TV Shows (~8,800 titles)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Goal:** Understand Netflix's content catalog — trends in genre, country, audience ratings, and growth over time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

## 1. Load & Initial Inspection

In [ ]:
df = pd.read_csv("netflix_titles.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Column names:", df.columns.tolist())
print()
df.info()

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

In [ ]:
# Visualise missing values
fig, ax = plt.subplots(figsize=(8, 4))
missing[missing > 0].plot(kind='bar', ax=ax, color='#E50914')
ax.set_title("Missing Values per Column")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Observations:**
- `director` and `cast` have the most missing values — acceptable for analysis since we focus on country, genre, rating.
- We drop rows missing `country` or `rating` as these are critical for key analyses.

## 3. Data Cleaning & Feature Engineering

In [ ]:
# Drop rows missing critical fields
df = df.dropna(subset=['country', 'rating'])
print(f"Rows after cleaning: {len(df)}")

In [ ]:
# Normalize date_added to datetime
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['year_added']  = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
print("date_added sample:")
df[['date_added', 'year_added', 'month_added']].head()

In [ ]:
# Feature: decade bins from release_year
df['decade'] = (df['release_year'] // 10) * 10
df['decade_label'] = df['decade'].astype(str) + 's'
df['decade_label'].value_counts().sort_index()

In [ ]:
# Feature: audience rating buckets
kids   = ['G', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'TV-G']
family = ['PG', 'TV-PG']
teen   = ['PG-13', 'TV-14']
adult  = ['R', 'TV-MA', 'NC-17', 'NR', 'UR']

def bucket_rating(r):
    if r in kids:   return 'Kids'
    if r in family: return 'Family'
    if r in teen:   return 'Teen'
    if r in adult:  return 'Adult'
    return 'Other'

df['rating_bucket'] = df['rating'].apply(bucket_rating)
df['rating_bucket'].value_counts()

In [ ]:
# Use primary country (first listed)
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()
print("Final dataframe shape:", df.shape)
df[['title','type','primary_country','release_year','decade_label','rating','rating_bucket']].head()

## 4. Content Type Analysis

In [ ]:
type_counts = df['type'].value_counts()
print(type_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar
sns.countplot(x='type', data=df, ax=axes[0], palette=['#E50914','#564d4d'])
axes[0].set_title("Movies vs TV Shows — Count")
axes[0].set_xlabel("Content Type")
axes[0].set_ylabel("Count")

# Pie
axes[1].pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
            colors=['#E50914','#564d4d'], startangle=90)
axes[1].set_title("Movies vs TV Shows — Share")

plt.tight_layout()
plt.show()

**Insight:** Netflix's catalog is ~**70% Movies** and ~**30% TV Shows**, reflecting its roots as a movie platform before its original content push.

## 5. Country Analysis

In [ ]:
top_countries = df['primary_country'].value_counts().head(10)
print(top_countries)

fig, ax = plt.subplots(figsize=(9, 4))
top_countries.plot(kind='barh', ax=ax, color='#E50914')
ax.invert_yaxis()
ax.set_title("Top 10 Countries Producing Netflix Content")
ax.set_xlabel("Number of Titles")
plt.tight_layout()
plt.show()

**Insight:** The **United States** dominates production (~35% of titles). **India** ranks second, reflecting Netflix's investment in Bollywood and regional content.

## 6. Content Growth Over Time

In [ ]:
# By release year
yearly = df['release_year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(yearly.index, yearly.values, alpha=0.3, color='#E50914')
ax.plot(yearly.index, yearly.values, color='#E50914', linewidth=2)
ax.set_title("Netflix Content Growth by Release Year")
ax.set_xlabel("Release Year")
ax.set_ylabel("Number of Titles")
ax.axvline(2015, linestyle='--', color='gray', label='2015 — Global Expansion')
ax.legend()
plt.tight_layout()
plt.show()

peak = int(yearly.idxmax())
print(f"Peak release year: {peak} ({yearly[peak]} titles)")

In [ ]:
# By year_added (when Netflix added it)
added_yearly = df['year_added'].value_counts().sort_index().dropna()

fig, ax = plt.subplots(figsize=(10, 4))
added_yearly.plot(kind='bar', ax=ax, color='#564d4d')
ax.set_title("Titles Added to Netflix by Year")
ax.set_xlabel("Year Added")
ax.set_ylabel("Number of Titles")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Insight:** Content additions surged post-2015 when Netflix launched internationally in 130+ countries. There's a noticeable **dip in 2020** (COVID-19 production disruptions).

## 7. Genre Analysis

In [ ]:
# Explode multi-label genre column
genres = df['listed_in'].str.split(',').explode().str.strip()
top_genres = genres.value_counts().head(15)
print(top_genres)

fig, ax = plt.subplots(figsize=(9, 5))
top_genres.plot(kind='barh', ax=ax, color='#E50914')
ax.invert_yaxis()
ax.set_title("Top 15 Genres on Netflix")
ax.set_xlabel("Number of Titles")
plt.tight_layout()
plt.show()

**Insight:** **Dramas** and **Comedies** dominate. International genres (International Movies, International TV Shows) rank high, highlighting Netflix's global strategy.

## 8. Decade & Audience Rating Analysis

In [ ]:
# Decade breakdown by content type
decade_type = df.groupby(['decade_label','type']).size().unstack(fill_value=0)
print(decade_type)

decade_type.plot(kind='bar', figsize=(10,4), color=['#E50914','#564d4d'])
plt.title("Content Count by Decade and Type")
plt.xlabel("Decade")
plt.ylabel("Number of Titles")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Rating bucket distribution
bucket_counts = df['rating_bucket'].value_counts()
print(bucket_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bucket_counts.plot(kind='bar', ax=axes[0], color=['#E50914','#b20710','#564d4d','#8b0000','#333'][:len(bucket_counts)])
axes[0].set_title("Content by Audience Rating Bucket")
axes[0].set_xlabel("Audience Group")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=0)

# By type
pd.crosstab(df['rating_bucket'], df['type']).plot(kind='bar', ax=axes[1], color=['#E50914','#564d4d'])
axes[1].set_title("Rating Bucket Split by Content Type")
axes[1].set_xlabel("Audience Group")
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

**Insight:** **Adult (TV-MA / R-rated)** content is the largest bucket, confirming Netflix targets mature audiences. Kids' content is proportionally small but exists across both movies and TV shows.

## 9. Summary of Findings

| Finding | Detail |
|---|---|
| Content split | ~70% Movies, ~30% TV Shows |
| Top producer | United States (~35% of catalog) |
| Most popular genre | Dramas, followed by Comedies |
| Growth inflection | 2015 — global expansion |
| Dominant audience | Adult (TV-MA / R) |
| Catalog recency | 70%+ of titles are from the 2010s or later |

**Next steps:** Sentiment analysis on descriptions, recommendation engine by genre/country, time-series forecasting of content additions.